# Phase 8 Worksheet — Evaluation Metrics
**Corrected in this version:** the LLM-as-judge section now uses `ask(..., model=MODEL_LLAMA)` which genuinely hits Llama's endpoint — under the old `multimodal_chat()`, passing `model=MODEL_LLAMA` silently still hit the Qwen3-14B endpoint, so any earlier "different judge model" comparison may not have actually been using a different model.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))  # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=200):
    """Drop-in replacement for the old multimodal_chat() text-only calls --
    correctly routed per-model via get_chat_model(), unlike inhouse_llm.py's
    own chat()/multimodal_chat() which always hit the Qwen3-14B endpoint."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=200):
    """Drop-in replacement for multimodal_chat() WITH an image -- uses the
    corrected image_url content-block format, and an actual client for the
    vision model (inhouse_llm.py never created one)."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

import chromadb
client = chromadb.HttpClient(host="localhost", port=8000)  # adjust to your Chroma server
print("Setup OK")

In [ ]:
collection = client.get_or_create_collection("phase8_eval")
docs = {f"doc_{i}": f"Document {i} about payment processing topic {i % 3}" for i in range(20)}
ids = list(docs.keys())
texts = list(docs.values())
collection.upsert(ids=ids, embeddings=embedder.embed_documents(texts), documents=texts)

# Gold set: for this query, assume these docs are "truly relevant" (topic 0)
gold_relevant = {f"doc_{i}" for i in range(20) if i % 3 == 0}
print("Total relevant docs in gold set:", len(gold_relevant))

## 1. Precision@K and Recall@K

In [ ]:
def precision_recall_at_k(query, k, gold_relevant):
    q_vec = embedder.embed_query(query)
    result = collection.query(query_embeddings=[q_vec], n_results=k)
    retrieved = set(result["ids"][0])
    relevant_retrieved = retrieved & gold_relevant
    precision = len(relevant_retrieved) / k
    recall = len(relevant_retrieved) / len(gold_relevant)
    return precision, recall

for k in [5, 10, 20]:
    p, r = precision_recall_at_k("payment processing topic 0", k, gold_relevant)
    print(f"K={k:2d}  Precision@K={p:.2f}  Recall@K={r:.2f}  (recall ceiling at this K = {min(k, len(gold_relevant))/len(gold_relevant):.2f})")

## 2. Reproducing this phase's teaser: low recall isn't always a bad retriever

In [ ]:
p5, r5 = precision_recall_at_k("payment processing topic 0", 5, gold_relevant)
print(f"Precision@5={p5:.2f} (looks fine) vs Recall@5={r5:.2f} (looks bad) -- check the ceiling before concluding anything.")

## 3. MRR

In [ ]:
def reciprocal_rank(query, gold_relevant, max_k=20):
    q_vec = embedder.embed_query(query)
    result = collection.query(query_embeddings=[q_vec], n_results=max_k)
    for rank, doc_id in enumerate(result["ids"][0], 1):
        if doc_id in gold_relevant:
            return 1 / rank
    return 0.0

queries = ["payment processing topic 0", "payment processing topic 1", "payment processing topic 2"]
gold_sets = [gold_relevant, {f"doc_{i}" for i in range(20) if i % 3 == 1}, {f"doc_{i}" for i in range(20) if i % 3 == 2}]
rrs = [reciprocal_rank(q, g) for q, g in zip(queries, gold_sets)]
print("Reciprocal ranks:", rrs)
print("MRR:", sum(rrs) / len(rrs))

## 4. Answer Faithfulness (LLM-as-judge, genuinely using a DIFFERENT model now)

In [ ]:
question = "What is doc_0 about?"
context = collection.get(ids=["doc_0"])["documents"][0]
answer = ask("Answer using only the context.", f"Context: {context}\nQuestion: {question}",
             model=MODEL_QWEN3_14B, max_tokens=100)

judge_prompt = f"Question: {question}\nContext: {context}\nAnswer: {answer}\n\nScore faithfulness 1-5 (does it ONLY use the context). Reply as JSON: {{\"score\": int, \"reason\": str}}"
verdict = ask("Be a strict evaluator. Reply with only JSON.", judge_prompt,
              model=MODEL_LLAMA, max_tokens=150)  # genuinely a different model now
print("Answer (from Qwen3-14B):", answer)
print("Faithfulness verdict (from Llama-70B):", verdict)

## Teaser exercise
Compute NDCG by hand for two rankings with the same Precision@5 but the single relevant document at different positions (position 1 vs position 5) — confirm NDCG gives a higher score to the ranking with the relevant doc earlier.